In [1]:
import pandas as pd
import numpy as np

df2 = pd.read_csv("../data/raw/districtproductionyield.csv")
print(df2.shape)
df2.info()
print("\nDuplicate rows:", df2.duplicated().sum())
print("\nNull counts:\n", df2.isnull().sum())

(575879, 8)
<class 'pandas.DataFrame'>
RangeIndex: 575879 entries, 0 to 575878
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   State_Name     575879 non-null  str    
 1   District_Name  575879 non-null  str    
 2   Crop_Year      575879 non-null  int64  
 3   Season         575879 non-null  str    
 4   Crop           575879 non-null  str    
 5   Area           575879 non-null  float64
 6   Production     575879 non-null  float64
 7   yield          575879 non-null  float64
dtypes: float64(3), int64(1), str(4)
memory usage: 35.1 MB

Duplicate rows: 0

Null counts:
 State_Name       0
District_Name    0
Crop_Year        0
Season           0
Crop             0
Area             0
Production       0
yield            0
dtype: int64


In [2]:
print("Unique states:", df2['State_Name'].nunique())
print("Unique districts:", df2['District_Name'].nunique())
print("Unique seasons:", df2['Season'].unique())
print("Unique crops:", df2['Crop'].nunique())
print("Year range:", df2['Crop_Year'].min(), "-", df2['Crop_Year'].max())

Unique states: 36
Unique districts: 730
Unique seasons: <StringArray>
['Kharif', 'Whole Year', 'Autumn', 'Rabi', 'Summer', 'Winter']
Length: 6, dtype: str
Unique crops: 128
Year range: 1997 - 2020


In [3]:
print(df2[['Area','Production','yield']].describe())

               Area    Production         yield
count  5.758790e+05  5.758790e+05  5.758790e+05
mean   1.186614e+04  5.689498e+07  4.779376e+03
std    4.407750e+04  1.656008e+09  7.109329e+04
min    8.000000e-03  0.000000e+00  0.000000e+00
25%    8.200000e+01  7.270000e+02  1.346025e+00
50%    5.870000e+02  1.220000e+04  4.484848e+01
75%    4.388000e+03  1.859000e+05  1.314807e+02
max    8.580100e+06  1.597800e+11  4.395833e+06


In [4]:
print(df2.groupby('Crop')['yield'].agg(['mean','median','max','count']).sort_values('max', ascending=False).head(15))

                            mean         median           max  count
Crop                                                                
Coconut            549978.962248  477758.318739  4.395833e+06   4715
Cashewnut             415.352025      24.149402  9.801000e+05   2586
Onion                 804.035784     500.000000  4.070000e+05  17600
Maize                 162.242224     112.500000  1.494000e+05  33757
Sannhamp               86.848022      25.000000  1.022000e+05   4808
Turmeric              234.874022      86.092715  9.648000e+04   9943
Sugarcane            3362.753817    2545.454545  8.800000e+04  18467
Potato                816.364510     634.414391  3.110196e+04  17428
Cotton(lint)          129.205214      50.000000  3.000000e+04  10650
Sesamum                30.935182      17.647059  2.750000e+04  21129
Arhar/Tur              52.956700      45.570866  2.700000e+04  18103
Banana               1769.149789    1033.303691  2.580690e+04   7497
Other Rabi pulses      75.095269  

In [5]:
print("Rows with zero production:", (df2['Production'] == 0).sum())
print("Rows with zero yield:", (df2['yield'] == 0).sum())
print(df2[df2['Production']==0][['State_Name','Crop','Crop_Year','Area']].head(10))

Rows with zero production: 3634
Rows with zero yield: 3634
          State_Name               Crop  Crop_Year    Area
395   Andhra Pradesh           Soyabean       2002    51.0
422   Andhra Pradesh           Cucumber       2002    18.0
426   Andhra Pradesh       other fibres       2002   132.0
428   Andhra Pradesh   Other Vegetables       2002  1767.0
479   Andhra Pradesh           Cucumber       2003    51.0
484   Andhra Pradesh   Other Vegetables       2003  1783.0
1196  Andhra Pradesh            Cabbage       2002     4.0
1201  Andhra Pradesh           Cucumber       2002     8.0
1206  Andhra Pradesh   Other Vegetables       2002   918.0
1208  Andhra Pradesh  Peas  (vegetable)       2002     1.0


In [6]:
print("Rows with zero/negative area:", (df2['Area'] <= 0).sum())

Rows with zero/negative area: 0


In [7]:
# check crop name overlap
target_crops = ['Rice', 'Maize', 'Cotton(lint)', 'Gram']
print(df2['Crop'].unique()[:50])  # scan for exact naming of chickpea/gram/cotton

<StringArray>
[                 'Arecanut',       'Other Kharif pulses',
                      'Rice',                    'Banana',
                 'Cashewnut',                   'Coconut',
                'Dry ginger',                 'Sugarcane',
              'Sweet potato',                   'Tapioca',
              'Black pepper',              'Dry chillies',
            'other oilseeds',                  'Turmeric',
                     'Maize',         'Moong(Green Gram)',
                      'Urad',                 'Arhar/Tur',
                 'Groundnut',                 'Sunflower',
                     'Bajra',               'Castor seed',
              'Cotton(lint)',                'Horse-gram',
                     'Jowar',                     'Korra',
                      'Ragi',                   'Tobacco',
                      'Gram',                     'Wheat',
                    'Masoor',                   'Sesamum',
                   'Linseed',             

In [8]:
crop_map = {
    'Rice': 'rice',
    'Maize': 'maize',
    'Cotton(lint)': 'cotton',
    'Gram': 'chickpea'
}

df2_filtered = df2[df2['Crop'].isin(crop_map.keys())].copy()
df2_filtered['Crop'] = df2_filtered['Crop'].map(crop_map)

print(df2_filtered.shape)
print(df2_filtered['Crop'].value_counts())

(97816, 8)
Crop
rice        36040
maize       33757
chickpea    17369
cotton      10650
Name: count, dtype: int64


In [9]:
print("Zero yield rows in filtered set:", (df2_filtered['yield'] == 0).sum())
df2_filtered = df2_filtered[df2_filtered['yield'] > 0].copy()
print("After dropping zero yield:", df2_filtered.shape)

Zero yield rows in filtered set: 144
After dropping zero yield: (97672, 8)


In [10]:
print(df2_filtered.groupby('Crop')['yield'].describe())

            count        mean          std       min       25%         50%  \
Crop                                                                         
chickpea  17328.0   53.456599    53.938519  0.016373  0.905028   52.000000   
cotton    10618.0  129.594606   574.341453  0.029412  1.674938   50.000000   
maize     33692.0  162.555228  1579.671790  0.001282  2.127298  112.830490   
rice      36034.0  126.645911   173.271673  0.003889  2.185327  108.994532   

                 75%            max  
Crop                                 
chickpea   93.032600    1308.333333  
cotton    189.491974   30000.000000  
maize     217.614260  149400.000000  
rice      224.184483   22372.727273  


In [11]:
# sanity check: is yield = Production / Area?
df2_filtered['calc_yield'] = df2_filtered['Production'] / df2_filtered['Area']
print(df2_filtered[['Crop','yield','calc_yield']].sample(10))
print("\nMax diff:", (df2_filtered['yield'] - df2_filtered['calc_yield']).abs().max
())

            Crop       yield  calc_yield
25946     cotton    0.375000    0.375000
393004      rice  228.571429  228.571429
463577     maize  259.248403  259.248403
484937      rice  210.526316  210.526316
345524     maize  133.333333  133.333333
570971      rice  197.535954  197.535954
502143  chickpea  159.176030  159.176030
74921       rice    0.717992    0.717992
302864     maize  122.605364  122.605364
134589    cotton    0.844186    0.844186

Max diff: 2.9103830456733704e-11


In [12]:
print(df2_filtered.nlargest(10, 'yield')[['Crop','State_Name','Crop_Year','Area','Production','yield']])

          Crop   State_Name  Crop_Year    Area  Production          yield
553572   maize  Maharashtra       1997     1.0    149400.0  149400.000000
553518   maize  Maharashtra       1997     3.0    398000.0  132666.666667
553604   maize  Maharashtra       1997     4.0    457000.0  114250.000000
553584   maize  Maharashtra       1997     5.0    563500.0  112700.000000
553428   maize  Maharashtra       1997     1.0    111300.0  111300.000000
553495   maize  Maharashtra       1997     4.0    290100.0   72525.000000
552848  cotton  Maharashtra       1997     2.0     60000.0   30000.000000
552856  cotton  Maharashtra       1997    45.0   1060000.0   23555.555556
552852  cotton  Maharashtra       1997    32.0    750000.0   23437.500000
554254    rice  Maharashtra       1997  1100.0  24610000.0   22372.727273


In [13]:
print("Rows with Area < 10:", (df2_filtered['Area'] < 10).sum())
print("Rows with Area < 50:", (df2_filtered['Area'] < 50).sum())

df2_filtered = df2_filtered[df2_filtered['Area'] >= 10].copy()
print("After filtering small area:", df2_filtered.shape)
print(df2_filtered.groupby('Crop')['yield'].describe())

Rows with Area < 10: 4711
Rows with Area < 50: 10812
After filtering small area: (92961, 9)
            count        mean         std       min       25%         50%  \
Crop                                                                        
chickpea  16334.0   53.826582   53.891539  0.016373  0.908022   53.072567   
cotton     9585.0  131.789966  510.576270  0.029412  1.751111   63.636364   
maize     31626.0  142.892004  169.066149  0.001282  2.148631  114.285714   
rice      35416.0  126.880803  174.134592  0.003889  2.193175  109.084268   

                 75%           max  
Crop                                
chickpea   93.176958   1308.333333  
cotton    197.247706  23555.555556  
maize     219.000262   2567.709065  
rice      224.723247  22372.727273  


In [14]:
# realistic yield bounds in the same unit as this dataset (appears to be kg/ha based on scale)
realistic_bounds = {
    'rice': (500, 8000),
    'maize': (500, 12000),
    'chickpea': (200, 3000),
    'cotton': (100, 3000)
}

def in_range(row):
    lo, hi = realistic_bounds[row['Crop']]
    return lo <= row['yield'] <= hi

mask = df2_filtered.apply(in_range, axis=1)
print("Rows kept:", mask.sum(), "out of", len(df2_filtered))
df2_final = df2_filtered[mask].copy()
print(df2_final.groupby('Crop')['yield'].describe())

Rows kept: 5689 out of 92961
           count        mean         std    min         25%         50%  \
Crop                                                                      
chickpea   113.0  249.204998  116.036282  200.0  206.682770  220.900322   
cotton    4206.0  236.209851  103.885967  100.0  153.846154  214.058801   
maize     1259.0  688.286784  186.503145  500.0  566.966256  660.000000   
rice       111.0  589.714656  192.603277  500.0  509.245268  530.000000   

                 75%          max  
Crop                               
chickpea  244.498778  1308.333333  
cotton    298.782450  1295.821980  
maize     750.926372  2567.709065  
rice      563.228959  1519.047619  


In [15]:
def clip_percentile(df, group_col, value_col, lower=0.02, upper=0.98):
    result = []
    for crop, group in df.groupby(group_col):
        lo, hi = group[value_col].quantile(lower), group[value_col].quantile(upper)
        filtered = group[(group[value_col] >= lo) & (group[value_col] <= hi)]
        result.append(filtered)
    return pd.concat(result)

df2_final = clip_percentile(df2_filtered, 'Crop', 'yield')
print("Rows kept:", len(df2_final), "out of", len(df2_filtered))
print(df2_final.groupby('Crop')['yield'].describe())


Rows kept: 89245 out of 92961
            count        mean         std       min       25%         50%  \
Crop                                                                        
chickpea  15680.0   51.777409   48.605145  0.344037  0.927428   53.072567   
cotton     9207.0  107.799329  119.690066  0.227273  1.836031   64.116274   
maize     30360.0  132.040455  138.778815  0.642157  2.253325  114.285714   
rice      33998.0  121.800035  118.686046  0.561644  2.251425  109.084268   

                 75%         max  
Crop                              
chickpea   91.699649  168.492633  
cotton    190.242876  450.000000  
maize     212.989240  658.798007  
rice      220.794611  410.663984  


In [16]:
# Use Dataset 2 only for state/district/crop/year -> season mapping
season_lookup = df2_filtered[['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop']].copy()
season_lookup.columns = season_lookup.columns.str.lower()
season_lookup = season_lookup.rename(columns={'crop_year': 'year'})
season_lookup = season_lookup.drop_duplicates()
print(season_lookup.shape)
print(season_lookup.head())

season_lookup.to_csv("../data/processed/dataset2_season_lookup.csv", index=False)
print("Saved dataset2_season_lookup.csv")

(56811, 5)
                     state_name district_name  year  season  crop
2   Andaman and Nicobar Islands      NICOBARS  2000  Kharif  rice
12  Andaman and Nicobar Islands      NICOBARS  2001  Kharif  rice
18  Andaman and Nicobar Islands      NICOBARS  2002  Kharif  rice
27  Andaman and Nicobar Islands      NICOBARS  2003  Kharif  rice
36  Andaman and Nicobar Islands      NICOBARS  2004  Kharif  rice
Saved dataset2_season_lookup.csv


In [ ]:
import pandas as pd
import numpy as np

df1 = pd.read_csv("../data/processed/dataset1_clean.csv")
season = pd.read_csv("../data/processed/dataset2_season_lookup.csv")

print(df1.shape)
print(season.shape)
print(df1.columns.tolist())
print(season.columns.tolist())